In [2]:
import pandas as pd

# Path to the Twitter Customer Support dataset
file_path = "../data/raw/twcs.csv"

# Load the dataset
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (2811774, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [3]:
# Look at some actual conversations
df[['tweet_id', 'inbound', 'text', 'response_tweet_id', 'in_response_to_tweet_id']].head(10)

,tweet_id,inbound,text,response_tweet_id,in_response_to_tweet_id
0,1,False,@115712 I understand. I would like to assist y...,2,3.0
1,2,True,@sprintcare and how do you propose we do that,NaN,1.0
2,3,True,@sprintcare I have sent several private messag...,1,4.0
3,4,False,@115712 Please send us a Private Message so th...,3,5.0
4,5,True,@sprintcare I did.,4,6.0
5,6,False,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,True,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,False,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,True,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,False,@115713 We understand your concerns and we'd l...,12,16.0


In [4]:
import re
from collections import Counter

# Get customer tweets only
customer_tweets = df[df["inbound"] == True]["text"].dropna()

# Extract @handles from customer tweets
handles = []

for text in customer_tweets:
    mentions = re.findall(r"@[\w]+", text)
    handles.extend(mentions)

# Count mentions
handle_counts = Counter(handles)

# Show the 30 most frequently mentioned handles
handle_counts.most_common(30)

[('@AmazonHelp', 136793),
 ('@AppleSupport', 98008),
 ('@AmericanAir', 50431),
 ('@Uber_Support', 47224),
 ('@Delta', 42461),
 ('@115858', 40695),
 ('@VirginTrains', 37458),
 ('@SouthwestAir', 34333),
 ('@Tesco', 33860),
 ('@SpotifyCares', 31213),
 ('@British_Airways', 31124),
 ('@XboxSupport', 28218),
 ('@GWRHelp', 27131),
 ('@ATVIAssist', 25129),
 ('@Ask_Spectrum', 23800),
 ('@115821', 23281),
 ('@sainsburys', 23105),
 ('@AskPlayStation', 22381),
 ('@TMobileHelp', 22093),
 ('@115873', 21864),
 ('@ChipotleTweets', 21807),
 ('@115850', 21671),
 ('@115911', 20734),
 ('@comcastcares', 19721),
 ('@O2', 19250),
 ('@hulu_support', 18792),
 ('@VerizonSupport', 17995),
 ('@Safaricom_Care', 16865),
 ('@idea_cares', 16612),
 ('@SW_Help', 15901)]

In [5]:
# Extract tweets that mention AppleSupport
apple_df = df[
    df["text"].str.contains("@AppleSupport", case=False, na=False)
].copy()

print("AppleSupport tweets:", len(apple_df))
print("Customer tweets:", (apple_df["inbound"] == True).sum())
print("Support tweets:", (apple_df["inbound"] == False).sum())

AppleSupport tweets: 97913
Customer tweets: 97896
Support tweets: 17


In [6]:
# Look at some AppleSupport customer messages
apple_customers = apple_df[apple_df["inbound"] == True]

apple_customers[["tweet_id", "created_at", "text"]].head(20)

,tweet_id,created_at,text
397,697,Tue Oct 31 22:31:23 +0000 2017,@AppleSupport The newest update. I️ made sure ...
399,698,Tue Oct 31 22:17:40 +0000 2017,@AppleSupport https://t.co/NV0yucs0lB
400,700,Tue Oct 31 22:16:56 +0000 2017,@AppleSupport why are my I️’s changing not sho...
402,702,Tue Oct 31 22:11:31 +0000 2017,@AppleSupport Tried resetting my settings .. r...
404,704,Tue Oct 31 21:59:17 +0000 2017,@AppleSupport This is what it looks like https...
406,707,Tue Oct 31 21:48:51 +0000 2017,@AppleSupport I️ have an iPhone 7 Plus and yes...
408,709,Tue Oct 31 21:34:45 +0000 2017,@AppleSupport I️ need answers because it’s ann...
412,713,Tue Oct 31 22:35:17 +0000 2017,@AppleSupport Just sent you all my DM
413,714,Tue Oct 31 22:19:32 +0000 2017,Hey @AppleSupport and anyone else who upgraded...
415,717,Tue Oct 31 22:13:08 +0000 2017,@AppleSupport This is what is happening... htt...


In [7]:
print("Total AppleSupport-related tweets:", len(apple_df))
print("Customer tweets:", (apple_df["inbound"] == True).sum())
print("Support tweets:", (apple_df["inbound"] == False).sum())

print("\nDate range:")
print("First:", apple_df["created_at"].min())
print("Last:", apple_df["created_at"].max())

Total AppleSupport-related tweets: 97913
Customer tweets: 97896
Support tweets: 17

Date range:
First: Fri Apr 14 02:12:01 +0000 2017
Last: Wed Sep 27 22:22:32 +0000 2017


In [8]:
# Get AppleSupport customer tweets
apple_customers = df[
    (df["inbound"] == True) &
    (df["text"].str.contains("@AppleSupport", case=False, na=False))
].copy()

# Get IDs of tweets that AppleSupport replied to
response_ids = []

for value in apple_customers["response_tweet_id"].dropna():
    # response_tweet_id can contain multiple IDs such as "123,456"
    ids = str(value).split(",")
    response_ids.extend([int(x.strip()) for x in ids if x.strip().isdigit()])

# Find those responses in the original dataset
apple_replies = df[
    df["tweet_id"].isin(response_ids) &
    (df["inbound"] == False)
].copy()

print("AppleSupport customer tweets:", len(apple_customers))
print("Potential AppleSupport replies:", len(apple_replies))

print("\nSample AppleSupport replies:")

print(apple_replies[["tweet_id", "text", "in_response_to_tweet_id"]].head(10).to_string(index=False))

AppleSupport customer tweets: 97896
Potential AppleSupport replies: 78959

Sample AppleSupport replies:
 tweet_id                                                                                                                                           text  in_response_to_tweet_id
      696                             @115854 We're here for you. Which version of the iOS are you running? Check from Settings &gt; General &gt; About.                    698.0
      699 @115854 Lets take a closer look into this issue. Select the following link to join us in a DM and we'll go from there. https://t.co/GDrqU22YpT                    697.0
      701                                                                 @115855 Let's go to DM for the next steps. DM us here: https://t.co/GDrqU22YpT                    702.0
      703                                                                                           @115855 Any steps tried since it started last night?                    704.0
      

In [9]:
# Pick one AppleSupport customer tweet
sample_tweet_id = apple_customers.iloc[0]["tweet_id"]

print("Starting tweet ID:", sample_tweet_id)

# Show the tweet
print("\nStarting tweet:")
print(
    df.loc[df["tweet_id"] == sample_tweet_id, "text"].iloc[0]
)

Starting tweet ID: 697

Starting tweet:
@AppleSupport The newest update. I️ made sure to download it yesterday.


In [10]:
# Show the conversation around this tweet

current_id = sample_tweet_id

for i in range(10):

    tweet = df[df["tweet_id"] == current_id]

    if tweet.empty:
        break

    row = tweet.iloc[0]

    print(f"\n--- Tweet {row['tweet_id']} ---")
    print("Inbound:", row["inbound"])
    print("Text:", row["text"])

    # Move to the next response
    response_ids = row["response_tweet_id"]

    if pd.isna(response_ids):
        break

    next_ids = str(response_ids).split(",")

    # Take the first response for now
    current_id = int(next_ids[0].strip())


--- Tweet 697 ---
Inbound: True
Text: @AppleSupport The newest update. I️ made sure to download it yesterday.

--- Tweet 699 ---
Inbound: False
Text: @115854 Lets take a closer look into this issue. Select the following link to join us in a DM and we'll go from there. https://t.co/GDrqU22YpT


In [11]:
# Find AppleSupport customer tweets that have multiple responses
apple_with_responses = apple_customers[
    apple_customers["response_tweet_id"].notna()
].copy()

# Count how many responses each customer tweet has
apple_with_responses["num_responses"] = (
    apple_with_responses["response_tweet_id"]
    .astype(str)
    .str.split(",")
    .str.len()
)

print("Customer tweets with responses:", len(apple_with_responses))

print("\nDistribution of number of responses:")
print(apple_with_responses["num_responses"].value_counts().sort_index())

Customer tweets with responses: 84439

Distribution of number of responses:
num_responses
1      76240
2       6626
3       1015
4        270
5         92
6         65
7         24
8         16
9         17
10        10
11        11
12         7
13         3
14         5
15         3
16         1
17         1
18         4
19         4
20         1
21         1
22         1
23         1
24         1
26         2
27         2
28         1
29         1
38         2
41         1
50         1
60         1
61         1
76         1
80         1
92         1
95         1
98         1
119        1
136        1
269        1
Name: count, dtype: int64


In [12]:
# Create customer → AppleSupport response pairs

pairs = []

for _, customer in apple_customers.iterrows():

    if pd.isna(customer["response_tweet_id"]):
        continue

    # Get all response IDs
    response_ids = str(customer["response_tweet_id"]).split(",")

    for response_id in response_ids:

        response_id = response_id.strip()

        if not response_id.isdigit():
            continue

        response_id = int(response_id)

        # Find the response tweet
        response = df[df["tweet_id"] == response_id]

        if response.empty:
            continue

        response = response.iloc[0]

        # Make sure it is actually a support response
        if response["inbound"] == False:

            pairs.append({
                "customer_tweet_id": customer["tweet_id"],
                "customer_message": customer["text"],
                "support_tweet_id": response["tweet_id"],
                "support_response": response["text"]
            })

# Convert to DataFrame
pairs_df = pd.DataFrame(pairs)

print("Total customer → support pairs:", len(pairs_df))

pairs_df.head(10)


Total customer → support pairs: 78959


,customer_tweet_id,customer_message,support_tweet_id,support_response
0,697,@AppleSupport The newest update. I️ made sure ...,699,@115854 Lets take a closer look into this issu...
1,698,@AppleSupport https://t.co/NV0yucs0lB,696,@115854 We're here for you. Which version of t...
2,702,@AppleSupport Tried resetting my settings .. r...,701,@115855 Let's go to DM for the next steps. DM ...
3,704,@AppleSupport This is what it looks like https...,703,@115855 Any steps tried since it started last ...
4,707,@AppleSupport I️ have an iPhone 7 Plus and yes...,705,@115855 That's great it has iOS 11.1 as we can...
5,709,@AppleSupport I️ need answers because it’s ann...,708,@115855 We'd like to look into this with you. ...
6,714,Hey @AppleSupport and anyone else who upgraded...,712,"@115856 Hey, let's work together to figure out..."
7,717,@AppleSupport This is what is happening... htt...,716,@115857 We'd like to investigate further with ...
8,721,@AppleSupport are the call centres closed for ...,720,@115859 We've received your DM and will contin...
9,723,@115858 @AppleSupport hello are all the lines ...,722,@115859 What's going on? We're hapy to help if...


In [13]:
# Check for missing values
print("Missing customer messages:")
print(pairs_df["customer_message"].isna().sum())

print("\nMissing support responses:")
print(pairs_df["support_response"].isna().sum())

print("\nDuplicate customer-support pairs:")
print(pairs_df.duplicated(
    subset=["customer_message", "support_response"]
).sum())

Missing customer messages:
0

Missing support responses:
0

Duplicate customer-support pairs:
0


In [14]:
# Basic text cleaning

pairs_clean = pairs_df.copy()

# Remove rows with missing text
pairs_clean = pairs_clean.dropna(
    subset=["customer_message", "support_response"]
)

# Remove exact duplicate pairs
pairs_clean = pairs_clean.drop_duplicates(
    subset=["customer_message", "support_response"]
)

# Remove extremely short messages
pairs_clean = pairs_clean[
    (pairs_clean["customer_message"].str.len() >= 10) &
    (pairs_clean["support_response"].str.len() >= 10)
]

# Reset index
pairs_clean = pairs_clean.reset_index(drop=True)

print("Original pairs:", len(pairs_df))
print("Clean pairs:", len(pairs_clean))
print("Removed:", len(pairs_df) - len(pairs_clean))

Original pairs: 78959
Clean pairs: 78959
Removed: 0


In [15]:
pairs_clean[[
    "customer_message",
    "support_response"
]].head(15)

,customer_message,support_response
0,@AppleSupport The newest update. I️ made sure ...,@115854 Lets take a closer look into this issu...
1,@AppleSupport https://t.co/NV0yucs0lB,@115854 We're here for you. Which version of t...
2,@AppleSupport Tried resetting my settings .. r...,@115855 Let's go to DM for the next steps. DM ...
3,@AppleSupport This is what it looks like https...,@115855 Any steps tried since it started last ...
4,@AppleSupport I️ have an iPhone 7 Plus and yes...,@115855 That's great it has iOS 11.1 as we can...
5,@AppleSupport I️ need answers because it’s ann...,@115855 We'd like to look into this with you. ...
6,Hey @AppleSupport and anyone else who upgraded...,"@115856 Hey, let's work together to figure out..."
7,@AppleSupport This is what is happening... htt...,@115857 We'd like to investigate further with ...
8,@AppleSupport are the call centres closed for ...,@115859 We've received your DM and will contin...
9,@115858 @AppleSupport hello are all the lines ...,@115859 What's going on? We're hapy to help if...


In [16]:
# Save cleaned AppleSupport customer-support pairs

output_path = "../data/processed/apple_support_pairs.csv"

pairs_clean.to_csv(output_path, index=False)

print(f"Saved successfully to: {output_path}")
print(f"Rows saved: {len(pairs_clean)}")

Saved successfully to: ../data/processed/apple_support_pairs.csv
Rows saved: 78959


In [17]:
# Verify saved dataset

import os

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size (MB):", round(os.path.getsize(output_path) / (1024 * 1024), 2))

File exists: True
File size (MB): 20.18


In [18]:
from collections import Counter
import re

# Combine all customer messages
texts = pairs_clean["customer_message"].astype(str).str.lower()

all_text = " ".join(texts)

# Extract words
words = re.findall(r"\b[a-z]{3,}\b", all_text)

# Common English words to ignore
stop_words = {
    "the", "and", "for", "you", "this", "that", "with",
    "have", "are", "was", "but", "not", "can", "what",
    "why", "how", "your", "just", "my", "its", "from",
    "they", "our", "out", "has", "get", "about",
    "apple", "support", "please", "help", "hey", "will",
    "would", "been", "when", "there", "does", "did",
    "who", "all", "any", "now", "too", "still", "just",
    "like", "know", "really", "got", "want", "need"
}

filtered_words = [
    word for word in words
    if word not in stop_words
]

word_counts = Counter(filtered_words)

# Display the 50 most common words
word_counts.most_common(50)

[('applesupport', 79574),
 ('iphone', 15482),
 ('phone', 13989),
 ('https', 12714),
 ('ios', 12396),
 ('update', 9799),
 ('fix', 7192),
 ('battery', 6138),
 ('new', 5542),
 ('app', 4383),
 ('since', 4263),
 ('after', 4196),
 ('screen', 3743),
 ('time', 3600),
 ('apps', 3398),
 ('updated', 3394),
 ('off', 3378),
 ('work', 3214),
 ('issue', 3114),
 ('back', 3015),
 ('amp', 2984),
 ('thanks', 2981),
 ('music', 2802),
 ('problem', 2711),
 ('don', 2669),
 ('even', 2664),
 ('every', 2648),
 ('doesn', 2642),
 ('working', 2584),
 ('only', 2459),
 ('one', 2339),
 ('keeps', 2321),
 ('yes', 2288),
 ('use', 2261),
 ('won', 2250),
 ('going', 2203),
 ('had', 2094),
 ('tried', 2077),
 ('plus', 1987),
 ('same', 1975),
 ('then', 1891),
 ('wifi', 1860),
 ('getting', 1852),
 ('last', 1802),
 ('using', 1799),
 ('having', 1781),
 ('latest', 1745),
 ('again', 1726),
 ('ipad', 1699),
 ('times', 1630)]

In [20]:
# Clean customer messages for topic discovery

def clean_for_topics(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove @mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove punctuation/numbers
    text = re.sub(r"[^a-z\s]", " ", text)

    # Split into words
    words = text.split()

    # Remove common words
    stop_words = {
        "the", "and", "for", "you", "this", "that", "with",
        "have", "are", "was", "but", "not", "can", "what",
        "why", "how", "your", "just", "my", "its", "from",
        "they", "our", "out", "has", "get", "about",
        "please", "help", "hey", "will", "would", "been",
        "when", "there", "does", "did", "who", "all",
        "any", "now", "too", "still", "like", "know",
        "really", "got", "want", "need", "thanks",
        "thank", "im", "ive", "dont", "doesnt", "cant",
        "could", "couldnt", "would", "also"
    }

    words = [
        word for word in words
        if word not in stop_words and len(word) > 2
    ]

    return words


# Create cleaned word list
topic_words = []

for text in pairs_clean["customer_message"]:
    topic_words.extend(clean_for_topics(text))

topic_counts = Counter(topic_words)

print("Top 50 topic-related words:\n")
print(topic_counts.most_common(50))

Top 50 topic-related words:

[('iphone', 16681), ('ios', 16097), ('phone', 14009), ('update', 9905), ('apple', 7462), ('fix', 7192), ('battery', 6139), ('new', 5541), ('app', 4382), ('since', 4263), ('after', 4198), ('screen', 3743), ('time', 3604), ('apps', 3398), ('updated', 3394), ('off', 3378), ('work', 3214), ('issue', 3115), ('back', 3015), ('amp', 2984), ('music', 2801), ('problem', 2712), ('don', 2669), ('even', 2664), ('every', 2651), ('doesn', 2642), ('working', 2583), ('plus', 2561), ('only', 2460), ('one', 2339), ('keeps', 2321), ('yes', 2288), ('use', 2261), ('won', 2250), ('going', 2203), ('had', 2095), ('tried', 2077), ('same', 1976), ('then', 1890), ('wifi', 1859), ('getting', 1852), ('last', 1804), ('using', 1799), ('having', 1781), ('latest', 1745), ('again', 1726), ('ipad', 1707), ('times', 1643), ('itunes', 1629), ('store', 1622)]


In [21]:
# Topics we want to investigate
topics = [
    "battery",
    "update",
    "iphone",
    "screen",
    "music",
    "app",
    "icloud",
    "password",
    "account",
    "charge"
]

# Show 5 random customer messages for each topic
for topic in topics:
    print("\n" + "=" * 80)
    print(f"TOPIC: {topic.upper()}")
    print("=" * 80)

    matches = pairs_clean[
        pairs_clean["customer_message"]
        .str.contains(topic, case=False, na=False)
    ]

    # Randomly sample up to 5 messages
    sample = matches.sample(
        min(5, len(matches)),
        random_state=42
    )

    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"\n{i}. {row['customer_message']}")


TOPIC: BATTERY

1. @AppleSupport hey guys can u plz check the health of my iPhone battery. It is draining very quickly. 20% down in 30 mins

2. @AppleSupport why with every iOS update the battery life is getting worse?

3. Iphone 8+ keeps freezing and cannot do hard restart.  WTF?  @applesupport 11.0.3 battery

4. @AppleSupport can y’all please fix the issue with Bluetooth turning itself on and draining the battery...it’s been like 5 updates...

5. @AppleSupport What is required to have Apple do a diagnostic on my iPhone 6 battery

TOPIC: UPDATE

1. @AppleSupport Horrible battery life and long charge time on my iPhone 7 since the new update. Anything I can do to fix it?

2. I am not happy with this iOS update😤 why can I no longer see my voicemails?! @AppleSupport

3. @AppleSupport And is this free? By the way I’ve already updated and I’m still having the same issues

4. @AppleSupport The others work. Just unable to download or update apps at this stage. Time and day are correct.

5. @

In [22]:
# Inspect additional topics

topics = [
    "icloud",
    "password",
    "charge",
    "refund",
    "music",
    "screen",
    "app"
]

for topic in topics:
    print("\n" + "=" * 80)
    print(f"TOPIC: {topic.upper()}")
    print("=" * 80)

    matches = pairs_clean[
        pairs_clean["customer_message"]
        .str.contains(topic, case=False, na=False)
    ]

    if len(matches) == 0:
        print("No matches found.")
        continue

    sample = matches.sample(
        min(5, len(matches)),
        random_state=42
    )

    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"\n{i}. {row['customer_message']}")


TOPIC: ICLOUD

1. @AppleSupport iPad is on 11.0.2 and iMac is 10.12.6. Same iCloud account on all devices

2. @AppleSupport Well, this is an iCloud + email thing so it affects all devices and their most current versions.

3. @AppleSupport why do I need to approve my iCloud from another fucking device after you ask for password &amp; pin code... y’all annoying

4. @AppleSupport How do I make it so that it’s saved on my device instead? And why can’t I retrieve them from iCloud with my network service on.

5. @AppleSupport my iphone x wont let me load a backup from the icloud. only has a delete option. help?

TOPIC: PASSWORD

1. @AppleSupport I can log in on my iPhone but password won't work on MacBook. I purchased album on phone but it doesn't show up in music library on phone.

2. @AppleSupport I've been trying to reset my Apple ID password for about a week now and it won't let me?

3. @AppleSupport I can not use my password in Notes app. I can open protected notes only using Touch ID.

In [23]:
from pathlib import Path
import sys

# Allow Python to import from the src directory
project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

from intent.taxonomy import INTENTS

print("Number of intents:", len(INTENTS))
print("\nSupported intents:")

for intent, details in INTENTS.items():
    print(f"- {intent}: {details['description']}")

Number of intents: 10

Supported intents:
- battery_power: Battery drain, battery health, overheating, or poor battery performance.
- software_update: Problems installing, downloading, or completing an iOS/software update.
- device_performance: Device freezing, crashing, restarting, becoming slow, or becoming unresponsive.
- charging_power: Problems with charging, chargers, cables, or the device not charging.
- apps: Problems with applications, including crashes, downloads, updates, or app functionality.
- account_icloud: Apple ID, iCloud, password, login, account access, or account-related problems.
- audio_media: Apple Music, voicemail, sound, audio playback, or other media-related problems.
- display_screen: Screen, display, touchscreen, brightness, or touch-response problems.
- billing_payment: Unexpected charges, payments, subscriptions, refunds, or billing-related issues.
- other_unknown: Requests that do not clearly fit another supported intent.
